# Portfolio Project #4

## E-Commerce Logistics & Delivery Performance

In [1]:
import pandas as pd
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)

In [2]:
# --- 1. Generate Warehouses ---
# Schema: warehouse_id, city, state 
warehouses_data = []
for i in range(1, 6): # 5 Warehouses
    warehouses_data.append({
        'warehouse_id': i,
        'city': fake.city(),
        'state': fake.state_abbr()
    })
df_warehouses = pd.DataFrame(warehouses_data)

In [3]:
# --- 2. Generate Drivers ---
# Schema: driver_id, name, vehicle_type 
vehicle_types = ['Van', 'Box Truck', 'Sprinter']
drivers_data = []
bad_driver_ids = [7, 12, 19] # These drivers will intentionally miss targets

for i in range(1, 31): # 30 Drivers
    drivers_data.append({
        'driver_id': i,
        'name': fake.name(),
        'vehicle_type': random.choice(vehicle_types)
    })
df_drivers = pd.DataFrame(drivers_data)

In [6]:
# --- 3. Generate Orders & Status Events ---
# Orders Schema: order_id, warehouse_id, weight_lbs 
# Status_Events Schema: event_id, order_id, driver_id, status, event_time 
orders_data = []
events_data = []
event_id_counter = 1

start_date = datetime.now() - timedelta(days=90) # 3-month period

for order_id in range(1, 5001): # 5,000 Orders
    warehouse_id = random.randint(1, 5)
    weight = round(random.uniform(0.5, 50.0), 2)
    
    orders_data.append({
        'order_id': order_id,
        'warehouse_id': warehouse_id,
        'weight_lbs': weight
    })
    
    # Timeline generation
    ordered_at = start_date + timedelta(days=random.randint(0, 85), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    
    # Event 1: Ordered
    events_data.append({'event_id': event_id_counter, 'order_id': order_id, 'driver_id': None, 'status': 'Ordered', 'event_time': ordered_at})
    event_id_counter += 1
    
    # Event 2: Picked (1 to 12 hours later)
    picked_at = ordered_at + timedelta(hours=random.randint(1, 12), minutes=random.randint(0, 59))
    events_data.append({'event_id': event_id_counter, 'order_id': order_id, 'driver_id': None, 'status': 'Picked', 'event_time': picked_at})
    event_id_counter += 1
    
    # Event 3: Shipped 
    # INJECTED DELAY: Warehouse 3 is severely bottlenecked
    if warehouse_id == 3:
        shipped_at = picked_at + timedelta(days=random.randint(3, 6)) # Takes days instead of hours
    else:
        shipped_at = picked_at + timedelta(hours=random.randint(2, 24))
        
    driver_id = random.randint(1, 30)
    events_data.append({'event_id': event_id_counter, 'order_id': order_id, 'driver_id': driver_id, 'status': 'Shipped', 'event_time': shipped_at})
    event_id_counter += 1

    # Event 4: Delivered
    # INJECTED DELAY: Specific drivers are terrible at hitting the 2-day delivery target
    if driver_id in bad_driver_ids:
        delivered_at = shipped_at + timedelta(days=random.randint(4, 8)) # Misses target
    else:
        delivered_at = shipped_at + timedelta(days=random.randint(1, 2), hours=random.randint(0, 12)) # Hits target
        
    events_data.append({'event_id': event_id_counter, 'order_id': order_id, 'driver_id': driver_id, 'status': 'Delivered', 'event_time': delivered_at})
    event_id_counter += 1

df_orders = pd.DataFrame(orders_data)
df_events = pd.DataFrame(events_data)

In [8]:
# --- 4. Export to CSV ---
df_warehouses.to_csv('warehouses.csv', index=False)
df_drivers.to_csv('drivers.csv', index=False)
df_orders.to_csv('orders.csv', index=False)
df_events.to_csv('status_events.csv', index=False)

print("Data generation complete! 4 CSV files have been created in your directory.")

Data generation complete! 4 CSV files have been created in your directory.
